## **Alumno:** <span style="color:red">**Juan Manuel Resquin**</span>
# <span style="color:red">**Entrenamiento de Modelos**</span>
### En este proceso, ya comenzare a subir los resultados del entrenamiento a MLflow, conectandolo no solo con GitHub, sino con DagsHub.v

### <span style="color:red">Este notebook usa churn_env para el registro oficial en MLflow y exportación del modelo .</span>

In [1]:
import pandas as pd
import numpy as np
import joblib
import mlflow
import dagshub
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, accuracy_score, precision_score, f1_score, cohen_kappa_score, classification_report

# Conexión con DagsHub y MLflow 
import dagshub
dagshub.init(repo_owner='JuanManuelResquin84', repo_name='Proyecto_AndesLink', mlflow=True)

# Volvemos a cargar el archivo, que ya sabemos que esta todo en orden 
df = pd.read_csv('../data/churn_sintetico.csv')  
print(df.head())

c:\Users\resqu\anaconda3\envs\churn_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Accessing as JuanManuelResquin84

Initialized MLflow to track repo "JuanManuelResquin84/Proyecto_AndesLink"

Repository JuanManuelResquin84/Proyecto_AndesLink initialized!

   tenure_months  monthly_charge  total_charges  support_tickets  \
0              7           58.23         326.50                2   
1             56           56.75        3154.21                0   
2             48           78.84        3864.31                3   
3             32           79.74        2511.40                0   
4             32           55.37        1735.51                3   

   late_payments  avg_monthly_usage_gb contract_type payment_method  \
0              1                 81.83       mensual  transferencia   
1              2                 96.52         anual         debito   
2              2                 93.60       bianual       efectivo   
3              0                 28.95       bianual         debito   
4              0                126.90         anual       efectivo   

  internet_service  has_streaming  has_security_pack  num_products  region  \
0            cable              0                  1             3  centro   
1       

# Preparación de los datos para el entrenamiento

In [2]:
df_ml = df.copy()

# Encoding para convertir los textos a números
# pd.get_dummies transformará automáticamente 'anual', 'mensual', etc., en columnas de 0 y 1
X = pd.get_dummies(df_ml.drop('churn', axis=1), drop_first=True)

# Variable Objetivo
# Usamos LabelEncoder para convertir 'Si/No' en 1/0.
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(df_ml['churn'])

# División del Dataset (80% para entrenamiento y 20% para test) y uso stratify para mantener la proporción de Churn en ambos sets
X_train_ml, X_test_ml, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Escalado para Naive Bayes y Random Forest

In [3]:
# Creamos las variables X_train y X_test definitivas y lo escalamos
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_ml)
X_test = scaler.transform(X_test_ml)

# Entrenamiento con el Modelo de Random Forest 

In [9]:
# Carga de Experimento al MLflow
with mlflow.start_run(run_name="RandomForest_1"):
    # Connfiguración del modelo con sus parámetros
    n_est1 = 100 
    m_depth1 = 5
    
    rf1_model = RandomForestClassifier(
        n_estimators=n_est1, 
        max_depth=m_depth1, 
        random_state=42,
        class_weight='balanced'
    )
    
    # Registro de parámetros en DagsHub
    mlflow.log_param("n_estimators", n_est1)
    mlflow.log_param("max_depth", m_depth1)
    mlflow.log_param("model_type", "Random Forest")
    
    # Entrenamiento Usando X_train (escalado) y y_train
    rf1_model.fit(X_train, y_train)
    
    # Predicción usando X_test escalado
    y_pred_rf1 = rf1_model.predict(X_test)
    
    # Métricas para el registro en DagsHub  
    # Si en tu split usaste y_test_ml, cambia y_test por y_test_ml abajo
    metrics_rf1 = {
        "accuracy": accuracy_score(y_test, y_pred_rf1),
        "precision": precision_score(y_test, y_pred_rf1),
        "recall": recall_score(y_test, y_pred_rf1),
        "f1_score": f1_score(y_test, y_pred_rf1),
        "kappa": cohen_kappa_score(y_test, y_pred_rf1)
    }
    
    mlflow.log_metrics(metrics_rf1)
    
    # Impresion de reporte detallado para visualización
    print("Métricas RandomForest_1:")
    print(classification_report(y_test, y_pred_rf1))

2026/05/02 11:28:34 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



Métricas RandomForest_1:
              precision    recall  f1-score   support

           0       0.81      0.65      0.72       660
           1       0.51      0.71      0.59       340

    accuracy                           0.67      1000
   macro avg       0.66      0.68      0.66      1000
weighted avg       0.71      0.67      0.68      1000

🏃 View run RandomForest_1 at: https://dagshub.com/JuanManuelResquin84/Proyecto_AndesLink.mlflow/#/experiments/0/runs/87ccdbc8868c49bbb6d3844938d39bef
🧪 View experiment at: https://dagshub.com/JuanManuelResquin84/Proyecto_AndesLink.mlflow/#/experiments/0


# Entrenamiento con el Modelo de Naive Bayes

In [22]:
# Definición de parámetros
umbral1 = 0.50

# Carga de experimento al MLflow
with mlflow.start_run(run_name="NaiveBayes_1"):
    
    nb1_model = GaussianNB()
    
    # Entrenamiento usando X_train escalado y y_train
    nb1_model.fit(X_train, y_train)
    
    # Ajuste de umbral, priorizando Recall para Churn
    y_probs_nb1 = nb1_model.predict_proba(X_test)[:, 1]
    y_pred_nb1 = (y_probs_nb1 >= umbral1).astype(int)
    
    # Cálculo de métricas, usando y_test para comparar
    acc = accuracy_score(y_test, y_pred_nb1)
    rec = recall_score(y_test, y_pred_nb1)
    prec = precision_score(y_test, y_pred_nb1)
    f1 = f1_score(y_test, y_pred_nb1)
    kappa = cohen_kappa_score(y_test, y_pred_nb1)
    
    # Logs para DagsHub
    mlflow.log_param("model_type", "GaussianNB")
    mlflow.log_param("probability_threshold", umbral1)
    
    # Registro de las métricas
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("recall", rec)
    mlflow.log_metric("precision", prec)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("kappa", kappa)
    
    # Guardado del modelo en MLflow
    mlflow.sklearn.log_model(nb1_model, "naive_bayes_andeslink1")
    
    # Resultados en consola con formato limpio
    print(f"Métricas NaiveBayes_1 (Umbral {umbral1}):")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print(f"Kappa:     {kappa:.4f}")
    
    # Resultado en Consola
    print(f"Métricas NaiveBayes_1 (Umbral {umbral1}):")
    print("-" * 30)
    print(classification_report(y_test, y_pred_nb1)) 
    print(f"Kappa: {kappa:.4f}")

2026/05/01 19:17:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/01 19:17:23 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Métricas NaiveBayes_1 (Umbral 0.5):
Accuracy:  0.6720
Recall:    0.6294
Precision: 0.5144
F1 Score:  0.5661
Kappa:     0.3067
Métricas NaiveBayes_1 (Umbral 0.5):
------------------------------
              precision    recall  f1-score   support

           0       0.78      0.69      0.74       660
           1       0.51      0.63      0.57       340

    accuracy                           0.67      1000
   macro avg       0.65      0.66      0.65      1000
weighted avg       0.69      0.67      0.68      1000

Kappa: 0.3067
🏃 View run NaiveBayes_1 at: https://dagshub.com/JuanManuelResquin84/Proyecto_AndesLink.mlflow/#/experiments/0/runs/7d91517fd65f46509b2bbdbed8f2436d
🧪 View experiment at: https://dagshub.com/JuanManuelResquin84/Proyecto_AndesLink.mlflow/#/experiments/0


In [11]:
# Definición de parámetros
umbral2 = 0.35

# Carga de experimento al MLflow
with mlflow.start_run(run_name="NaiveBayes_2"):
    
    nb2_model = GaussianNB()
    
    # Entrenamiento usando X_train escalado y y_train
    nb2_model.fit(X_train, y_train)
    
    # Ajuste de umbral, priorizando Recall para Churn
    y_probs_nb2 = nb2_model.predict_proba(X_test)[:, 1]
    y_pred_nb2 = (y_probs_nb2 >= umbral2).astype(int)
    
    # Cálculo de métricas, usando y_test para comparar
    acc = accuracy_score(y_test, y_pred_nb2)
    rec = recall_score(y_test, y_pred_nb2)
    prec = precision_score(y_test, y_pred_nb2)
    f1 = f1_score(y_test, y_pred_nb2)
    kappa = cohen_kappa_score(y_test, y_pred_nb2)
    
    # Logs para DagsHub
    mlflow.log_param("model_type", "GaussianNB")
    mlflow.log_param("probability_threshold", umbral2)
    
    # Registro de las métricas
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("recall", rec)
    mlflow.log_metric("precision", prec)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("kappa", kappa)
    
    # Guardado del modelo en MLflow
    mlflow.sklearn.log_model(nb2_model, "naive_bayes_andeslink2")
    
    # Resultados en consola con formato limpio
    print(f"Métricas NaiveBayes_2 (Umbral {umbral2}):")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print(f"Kappa:     {kappa:.4f}")
    
        # Resultado en Consola
    print(f"Métricas NaiveBayes_2 (Umbral {umbral2}):")
    print("-" * 30)
    print(classification_report(y_test, y_pred_nb2)) 
    print(f"Kappa: {kappa:.4f}")

2026/05/02 11:30:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 11:30:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Métricas NaiveBayes_2 (Umbral 0.35):
Accuracy:  0.6360
Recall:    0.7912
Precision: 0.4786
F1 Score:  0.5965
Kappa:     0.2998
Métricas NaiveBayes_2 (Umbral 0.35):
------------------------------
              precision    recall  f1-score   support

           0       0.84      0.56      0.67       660
           1       0.48      0.79      0.60       340

    accuracy                           0.64      1000
   macro avg       0.66      0.67      0.63      1000
weighted avg       0.72      0.64      0.64      1000

Kappa: 0.2998
🏃 View run NaiveBayes_2 at: https://dagshub.com/JuanManuelResquin84/Proyecto_AndesLink.mlflow/#/experiments/0/runs/313ef0c1bd324b318ea714ab30429737
🧪 View experiment at: https://dagshub.com/JuanManuelResquin84/Proyecto_AndesLink.mlflow/#/experiments/0


# <span style="color:red">**Conclusión**</span>

### Para un problema de Churn, yo me quedaría con el NaiveBayes_2, basándome en los sigueintes datos:

* **El Recall:** el modelo NaiveBayes_2 tiene un Recall de 0.7911 frente al 0.65 del RandomForest_3. Esto significa que el Naive Bayes detecta a 269 clientes que realmente se iban a ir, mientras que el Bosque solo detecta a 221. En un negocio, esos 48 clientes extra recuperados valen mucho dinero.

* **La Matriz de Confusión:** observando los gráfico el Naive Bayes tiene menos "Falsos Negativos" (solo 71) comparado con los 119 del Bosque. Es preferible llamar a alguien que no se iba a ir (Falso Positivo) que ignorar a alguien que está por abandonar la empresa.

* **Estabilidad del Kappa:** ambos están cerca del 0.30, lo cual es aceptable para un dataset de este tamaño de 5000 filas.

* **Sobre el Accuracy:** el Accuracy difícilmente va a mejorar mucho más, y de hecho, no debería ser la métrica principal. En problemas de Churn, el dataset suele estar desbalanceado. Si el 80% de la gente se queda, un modelo que diga que nadie se va, tendría un 80% de Accuracy pero sería inútil para el negocio.

**Costo de oportunidad:** el RandomForest_3 tiene mejor Accuracy (0.693 vs 0.636) pero es más conservador y falla menos al predecir quiénes se quedan, pero a costa de perder clientes que sí se van.

### Por todo lo evaluado si bien el RandomForest_3 como el modelo es más equilibrado con mejor precisión y f1-score, concluyo que para el objetivo de AndesLink, el NaiveBayes_2 es el ganador debido a su capacidad superior de detectar con el Recall.
### Se sacrifica un poco de Accuracy y Precision en favor de un Recall del 79%, asegurando que la mayor cantidad de clientes en riesgo sean identificados para campañas de fidelización".

# <span style="color:red">**Empaquetado del Modelo y Escalador para Producción**</span> 
### Al ejecutar estas líneas, estoy asegurandome la reproducibilidad del modelo. No solo guardando el modelo Naive Bayes, sino también los el scaler y las columnas, lo cual es indispensable para que el modelo funcione correctamente fuera de tu entorno de entrenamiento.

In [14]:
import joblib
from sklearn.naive_bayes import GaussianNB

# Entrenamiento
model_nb2 = GaussianNB()
model_nb2.fit(X_train, y_train)

# Ruta completa hacia tu carpeta 'models' según tu estructura
base_path = r'E:/CIENCIA DE DATOS E IA/2-SEGUNDO AÑO/SEGUNDO CUATRIMESTRE/2- LABORATORIO DE MINERIA Y DATOS/Proyecto_AndesLink/models/'
modelo_file = base_path + 'modelo_churn_nbV1_andeslink.pkl'
scaler_file = base_path + 'scaler_andeslink.pkl'
columnas_file = base_path + 'X_columns.pkl'

# Guardamos el modelo serializado
joblib.dump(model_nb2, modelo_file)
joblib.dump(scaler, scaler_file) # El scaler usado
joblib.dump(X.columns.tolist(), columnas_file) # El X con dummies

print(f"Listo")

Listo
